# Lesson 3: Chatbot Example

- Practice for Lesson 3: Chatbot Example

## References
- [arxiv](https://pypi.org/project/arxiv/)
- [OpenAI Docs](https://platform.openai.com/docs/quickstart?api-mode=chat)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling?api-mode=chat)
- [OpenAI Chat Completions API Reference](https://platform.openai.com/docs/api-reference/chat/create)
- [Google Gemini Docs](https://ai.google.dev/gemini-api/docs/openai)

## Install Dependencies

In [ ]:
%pip install arxiv
%pip install openai

## Import Libraries

In [ ]:
import arxiv
import json
import os
from typing import List
from openai import OpenAI

## Tool Functions

In [28]:
PAPER_DIR = "papers"
PAPER_INFO_FILE = "papers_info.json"

In [29]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.

    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)

    Returns:
        List of paper IDs found in the search
    """

    # Get Papers from arXiv
    client = arxiv.Client()
    search = arxiv.Search(
        query=topic, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)

    # Create directory for the path
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)

    # File Path
    filePath = os.path.join(path, PAPER_INFO_FILE)

    # Try to load existing papers info
    try:
        with open(filePath, "r") as json_file:
            paper_infos = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        paper_infos = {}

    # Process each papaer and add it to paper_infos
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "summary": paper.summary,
            "pdf_url": paper.pdf_url,
            "published": str(paper.published.date()),
        }
        paper_infos[paper.get_short_id()] = paper_info

    # Save updated paper_info to the file
    with open(filePath, "w") as json_file:
        json.dump(paper_infos, json_file, indent=2)

    return paper_ids

In [30]:
search_papers("computers")

['1310.7911v2',
 'math/9711204v1',
 '2208.00733v1',
 '2504.07020v1',
 '2403.03925v1']

In [31]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """

    for item in os.listdir(PAPER_DIR):
        topic_dir_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(topic_dir_path):
            # print(f"Topic Directory : {item}")

            paper_file_path = os.path.join(topic_dir_path, PAPER_INFO_FILE)
            if os.path.isfile(paper_file_path):
                
                # Try to load existing papers info
                try:
                    with open(paper_file_path, "r") as json_file:
                        paper_infos = json.load(json_file)
                        # print(f"Paper Info - {paper_infos}")
                except (FileNotFoundError, json.JSONDecodeError):
                    paper_infos = {}
                
                # Check if the given paper is present
                for id, paper in paper_infos.items():
                    if id == paper_id:
                        return json.dumps(paper, indent=2)
                    

    return f"There's no saved information related to paper {paper_id}"

In [32]:
extract_info("1310.7911v2")

'{\n  "title": "Compact manifolds with computable boundaries",\n  "authors": [\n    "Zvonko Iljazovic"\n  ],\n  "summary": "We investigate conditions under which a co-computably enumerable closed set\\nin a computable metric space is computable and prove that in each locally\\ncomputable computable metric space each co-computably enumerable compact\\nmanifold with computable boundary is computable. In fact, we examine the notion\\nof a semi-computable compact set and we prove a more general result: in any\\ncomputable metric space each semi-computable compact manifold with computable\\nboundary is computable. In particular, each semi-computable compact\\n(boundaryless) manifold is computable.",\n  "pdf_url": "http://arxiv.org/pdf/1310.7911v2",\n  "published": "2013-10-29"\n}'

## Tool Schema

In [33]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search for papers on arXiv based on a topic and store their information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search for",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to retrieve",
                        "default": 5,
                    },
                },
                "required": ["topic"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_info",
            "description": "Search for information about a specific paper across all topic directories.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "The ID of the paper to look for",
                    },
                },
                "required": ["paper_id"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

## Tool Mapping

In [34]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    # execute tool
    result = mapping_tool_function[tool_name](**tool_args)

    if result == None:
        result = "The operation completed but didn't return any results."
    
    elif isinstance(result, list):
        result = ', '.join(result)

    elif isinstance(result, dict):
        json.dumps(result, indent=2)

    else:
        result = str(result)

    return result

## Chatbot Code

In [35]:
client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
model = "gemini-2.5-flash"

In [36]:
def process_query(query):
    messages = [{"role": "user", "content": query}]

    # Call Model
    completion = client.chat.completions.create(
        model=model, messages=messages, tools=tools
    )
    message = completion.choices[0].message
    # print(f"Message: {message}")

    process_query = True
    while process_query:
        # Append the assistant message
        messages.append(message)

        # Check if the message contains tool calls
        if message.tool_calls:
            tool_call = message.tool_calls[0]

            tool_id = tool_call.id
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"Calling tool {tool_name} with args {tool_args}")
            # Execute the tool
            tool_result = execute_tool(tool_name, tool_args)
            # print(f"Tool result: {tool_result}")

            # Append the tool result to the messages
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_id,
                    "content": str(tool_result),
                }
            )

            # Call the model again with the updated messages
            completion = client.chat.completions.create(
                model=model, messages=messages, tools=tools
            )
            message = completion.choices[0].message
            # print(f"Message: {message}")
        else:
            # No tool calls, we can stop processing
            print(message.content)
            process_query = False


### Chat Loop

In [37]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

In [38]:
chat_loop()

Type your queries or 'quit' to exit.
